In [1]:
!pip -q install -U transformers datasets accelerate peft trl sentence-transformers scikit-learn pandas matplotlib bitsandbytes
import torch, os, random, numpy as np, pandas as pd
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Select Runtime > Change runtime type > GPU before training.")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 13.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not cur

In [2]:
from google.colab import files
import io

uploaded = files.upload()  # Upload your English CSV
csv_name = next(iter(uploaded))
raw_df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
print("File:", csv_name, "| Shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
display(raw_df.head())

Saving FakeNewsNet.csv to FakeNewsNet.csv
File: FakeNewsNet.csv | Shape: (23196, 5)
Columns: ['title', 'news_url', 'source_domain', 'tweet_num', 'real']


,title,news_url,source_domain,tweet_num,real
0,Kandi Burruss Explodes Over Rape Accusation on...,http://toofab.com/2017/05/08/real-housewives-a...,toofab.com,42,1
1,People's Choice Awards 2018: The best red carp...,https://www.today.com/style/see-people-s-choic...,www.today.com,0,1
2,Sophia Bush Sends Sweet Birthday Message to 'O...,https://www.etonline.com/news/220806_sophia_bu...,www.etonline.com,63,1
3,Colombian singer Maluma sparks rumours of inap...,https://www.dailymail.co.uk/news/article-33655...,www.dailymail.co.uk,20,1
4,Gossip Girl 10 Years Later: How Upper East Sid...,https://www.zerchoo.com/entertainment/gossip-g...,www.zerchoo.com,38,1


In [4]:
TEXT_COL = "title"       # Updated to match FakeNewsNet.csv
LABEL_COL = "real"      # Updated to match FakeNewsNet.csv

LABEL_MAP = {
    "fake": 0, "false": 0, "0": 0,
    "real": 1, "true": 1, "1": 1
}

print(raw_df[LABEL_COL].value_counts(dropna=False))

real
1    17441
0     5755
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

def map_label(x):
    key = str(x).strip().lower()
    if key not in LABEL_MAP:
        raise ValueError(f"Unmapped label {x!r}. Update LABEL_MAP.")
    return LABEL_MAP[key]

if TEXT_COL not in raw_df.columns or LABEL_COL not in raw_df.columns:
    raise ValueError(f"Check TEXT_COL/LABEL_COL. Available columns: {raw_df.columns.tolist()}")

df = raw_df[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL:"text", LABEL_COL:"raw_label"}).copy()
df = df.dropna(subset=["text", "raw_label"])
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0]
df["label"] = df["raw_label"].apply(map_label).astype(int)
df = df[["text", "label"]].drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

train_val, test_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=SEED)
train_df, val_df = train_test_split(train_val, test_size=0.1765, stratify=train_val["label"], random_state=SEED)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for name, part in [("train",train_df),("validation",val_df),("test",test_df)]:
    print(name, part.shape, part["label"].value_counts().to_dict())
display(train_df.head())

train (15291, 2) {1: 11566, 0: 3725}
validation (3278, 2) {1: 2479, 0: 799}
test (3278, 2) {1: 2479, 0: 799}


,text,label
0,Nicole Kidman is helping Isabella Cruise adopt...,0
1,Eric Stonestreet Wrestles Ellen DeGeneres' Pra...,1
2,Selena Gomez Debuted A New Look on the AMA Red...,0
3,Spirit Awards 2018 Winners List – Variety,1
4,Justin Bieber Despacito,0


In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Multilingual model can also support Arabic datasets.
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)

train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
train_embeddings = embedder.encode(
    train_texts, batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
print("Retrieval index embeddings:", train_embeddings.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/478 [00:00<?, ?it/s]

Retrieval index embeddings: (15291, 384)


In [8]:
# A cross-encoder reranks (query, candidate) pairs.
# It scores semantic relevance, not truthfulness.
from sentence_transformers import CrossEncoder

RERANKER_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
reranker = CrossEncoder(RERANKER_NAME, max_length=256)

TOP_CANDIDATES = 20
TOP_K = 4

def retrieve_and_rerank(query, top_candidates=TOP_CANDIDATES, top_k=TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    sims = (train_embeddings @ q_emb[0])
    candidate_ids = np.argsort(sims)[::-1][:min(top_candidates, len(train_df))]
    candidates = [train_texts[i] for i in candidate_ids]
    pairs = [(query, c) for c in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)
    order = np.argsort(scores)[::-1][:min(top_k, len(candidate_ids))]
    result = []
    for rank, j in enumerate(order, start=1):
        idx = int(candidate_ids[j])
        result.append({
            "rank": rank,
            "text": train_texts[idx],
            "label": int(train_labels[idx]),
            "retrieval_similarity": float(sims[idx]),
            "rerank_score": float(scores[j])
        })
    return result

# Quick retrieval sanity check
sample_query = test_df.iloc[0]["text"]
for item in retrieve_and_rerank(sample_query):
    print(f"Rank {item['rank']} | train label={item['label']} | rerank={item['rerank_score']:.3f}")
    print(item["text"][:350], "\n")

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Rank 1 | train label=0 | rerank=3.195
Caitlyn Jenner wants a baby with Kim Kardashian's surrogate 

Rank 2 | train label=0 | rerank=2.349
Kim Kardashian, Khloe and Kourtney 'distancing themselves from Bruce Jenner' 

Rank 3 | train label=1 | rerank=1.754
Caitlyn Jenner Opens Up About ‘Tough’ Relationship With Her Kids 

Rank 4 | train label=1 | rerank=1.034
Kourtney and Kim Kardashian Hit the Slopes with Kendall Jenner for a Family Ski Trip 



In [9]:
def build_prompt(text, retrieved, include_labels=True):
    context_lines = []
    for i, item in enumerate(retrieved, start=1):
        lab = {0:"Fake", 1:"Real"}[item["label"]]
        if include_labels:
            context_lines.append(f"Example {i} (training label: {lab}): {item['text']}")
        else:
            context_lines.append(f"Related training text {i}: {item['text']}")
    contexts = "\n".join(context_lines) if context_lines else "No related training examples were retrieved."

    return f'''You are a careful fake-news classification assistant.
Classify the post as Fake or Real based on the post text and the similar labelled training examples.
The retrieved examples are similarity-based training examples, not independently verified evidence.
Do not invent facts. If the text is insufficient, choose the most plausible label but explain uncertainty briefly.

SIMILAR TRAINING EXAMPLES:
{contexts}

POST TO CLASSIFY:
{text}

Return exactly this format:
Label: Fake or Real
Reason: one short sentence
'''

In [ ]:
def retrieve_for_training_row(row_idx, top_k=TOP_K):
    query = train_texts[row_idx]
    q_emb = train_embeddings[row_idx]
    sims = train_embeddings @ q_emb
    sims[row_idx] = -1e9  # exclude the same training row
    candidate_ids = np.argsort(sims)[::-1][:min(TOP_CANDIDATES, len(train_df)-1)]
    candidates = [train_texts[i] for i in candidate_ids]
    scores = reranker.predict([(query, c) for c in candidates], show_progress_bar=False)
    order = np.argsort(scores)[::-1][:min(top_k, len(candidate_ids))]
    return [{
        "text": train_texts[int(candidate_ids[j])],
        "label": int(train_labels[int(candidate_ids[j])]),
        "rerank_score": float(scores[j])
    } for j in order]

def to_training_text(text, label, retrieved):
    prompt = build_prompt(text, retrieved)
    target = "Fake" if int(label) == 0 else "Real"
    return {"text": prompt + f"\nLabel: {target}\nReason: Classification based on the labelled dataset example."}

# Use a limit first to test pipeline and GPU memory; set None to use all training rows.
MAX_TRAIN_ROWS = None  # e.g., 100 for a quick smoke test
MAX_VAL_ROWS = None

train_subset = train_df if MAX_TRAIN_ROWS is None else train_df.iloc[:MAX_TRAIN_ROWS]
val_subset = val_df if MAX_VAL_ROWS is None else val_df.iloc[:MAX_VAL_ROWS]

train_records = []
for idx in range(len(train_subset)):
    # Current version expects the full train_df order. If MAX_TRAIN_ROWS is set, first rows align.
    retrieved = retrieve_for_training_row(idx)
    train_records.append(to_training_text(train_df.iloc[idx]["text"], train_df.iloc[idx]["label"], retrieved))

val_records = []
for _, row in val_subset.iterrows():
    retrieved = retrieve_and_rerank(row["text"])
    val_records.append(to_training_text(row["text"], row["label"], retrieved))

print("Training records:", len(train_records), "| Validation records:", len(val_records))
print(train_records[0]["text"][:1200])

In [1]:
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"  # or "Qwen/Qwen2.5-3B-Instruct"
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_4bit = torch.cuda.is_available()
bnb_config = None
if use_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
if use_4bit:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"
)

train_hf = Dataset.from_list(train_records)
val_hf = Dataset.from_list(val_records)

sft_args = SFTConfig(
    output_dir="/content/rankrag_llm_output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    bf16=False,
    max_length=1024,
    packing=False,
    report_to="none",
    seed=SEED
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    peft_config=lora_config,
    processing_class=tokenizer
)

trainer.train()

ModuleNotFoundError: No module named 'trl'